In [1]:
# Cell 1 - Configurar rutas
lakehouses = mssparkutils.lakehouse.list()
for lh in lakehouses:
    if 'Silver' in lh['displayName']:
        SILVER_ID = lh['id']
        WORKSPACE_ID = lh['workspaceId']
    if 'Gold' in lh['displayName']:
        GOLD_ID = lh['id']

SILVER_PATH = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{SILVER_ID}"
GOLD_PATH = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{GOLD_ID}"

print(f"Silver: {SILVER_ID}")
print(f"Gold: {GOLD_ID}")
print("Rutas configuradas OK")

StatementMeta(, 4545488f-96cd-4948-8e44-92251a8e5410, 3, Finished, Available, Finished, False)

Silver: 7238c65b-dcce-476a-9311-134ae8788a34
Gold: c3518afb-2909-426d-9726-991744a55153
Rutas configuradas OK


In [2]:
# Cell 2 - Crear DimDate con ISO Calendar completo
import pandas as pd

date_range = pd.date_range(start='2010-01-01', end='2026-12-31', freq='D')

def get_iso_period(iso_week):
    """Convierte ISO week a período (1-13, cada 4 semanas)"""
    return ((iso_week - 1) // 4) + 1

date_df = pd.DataFrame({
    'date_key':        [int(d.strftime('%Y%m%d')) for d in date_range],
    'date':            date_range,
    'year':            [d.year for d in date_range],
    'quarter':         [d.quarter for d in date_range],
    'quarter_name':    [f"Q{d.quarter}" for d in date_range],
    'year_quarter':    [f"{d.year}-Q{d.quarter}" for d in date_range],
    'month':           [d.month for d in date_range],
    'month_name':      [d.strftime('%b') for d in date_range],
    'month_name_full': [d.strftime('%B') for d in date_range],
    'year_month':      [d.strftime('%Y-%m') for d in date_range],
    'day':             [d.day for d in date_range],
    'day_of_week':     [d.isoweekday() for d in date_range],
    'day_name':        [d.strftime('%a') for d in date_range],
    'day_name_full':   [d.strftime('%A') for d in date_range],
    'is_weekend':      [1 if d.isoweekday() >= 6 else 0 for d in date_range],
    # ISO Calendar
    'iso_year':        [d.isocalendar()[0] for d in date_range],
    'iso_week':        [d.isocalendar()[1] for d in date_range],
    'iso_year_week':   [f"{d.isocalendar()[0]}-W{d.isocalendar()[1]:02d}" for d in date_range],
    # ISO Periods (13 períodos de 4 semanas)
    'iso_period':      [get_iso_period(d.isocalendar()[1]) for d in date_range],
    'iso_period_name': [f"P{get_iso_period(d.isocalendar()[1]):02d}" for d in date_range],
    'iso_year_period': [f"{d.isocalendar()[0]}-P{get_iso_period(d.isocalendar()[1]):02d}" for d in date_range],
})

dim_date = spark.createDataFrame(date_df)
dim_date.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("DimDate")
print(f"OK: DimDate ({dim_date.count():,} filas, {len(dim_date.columns)} columnas)")
print(f"Columnas: {dim_date.columns}")

# Verificar ISO periods
print("\nEjemplo semanas vs períodos:")
dim_date.filter((dim_date.iso_year == 2024) & (dim_date.iso_week <= 12)) \
    .select("date", "iso_week", "iso_period", "iso_year_period", "iso_year_week") \
    .distinct() \
    .orderBy("date") \
    .show(12)

StatementMeta(, f4005b0e-be25-4b2c-b241-9cbf8280ec6c, 4, Finished, Available, Finished, False)

OK: DimDate (6,209 filas, 21 columnas)
Columnas: ['date_key', 'date', 'year', 'quarter', 'quarter_name', 'year_quarter', 'month', 'month_name', 'month_name_full', 'year_month', 'day', 'day_of_week', 'day_name', 'day_name_full', 'is_weekend', 'iso_year', 'iso_week', 'iso_year_week', 'iso_period', 'iso_period_name', 'iso_year_period']

Ejemplo semanas vs períodos:
+-------------------+--------+----------+---------------+-------------+
|               date|iso_week|iso_period|iso_year_period|iso_year_week|
+-------------------+--------+----------+---------------+-------------+
|2024-01-01 00:00:00|       1|         1|       2024-P01|     2024-W01|
|2024-01-02 00:00:00|       1|         1|       2024-P01|     2024-W01|
|2024-01-03 00:00:00|       1|         1|       2024-P01|     2024-W01|
|2024-01-04 00:00:00|       1|         1|       2024-P01|     2024-W01|
|2024-01-05 00:00:00|       1|         1|       2024-P01|     2024-W01|
|2024-01-06 00:00:00|       1|         1|       2024-P01|  

In [2]:
# Cell 3 - DimProduct
from pyspark.sql.functions import col

df_product = spark.read.format("delta").load(f"{SILVER_PATH}/Tables/silver_production_product")
df_subcat = spark.read.format("delta").load(f"{SILVER_PATH}/Tables/silver_production_productsubcategory")
df_cat = spark.read.format("delta").load(f"{SILVER_PATH}/Tables/silver_production_productcategory")

df_subcat_clean = df_subcat.select(
    col("product_subcategory_id"),
    col("name").alias("subcategory_name"),
    col("product_category_id")
)

df_cat_clean = df_cat.select(
    col("product_category_id"),
    col("name").alias("category_name")
)

dim_product = df_product.join(df_subcat_clean, on="product_subcategory_id", how="left") \
    .join(df_cat_clean, on="product_category_id", how="left") \
    .select(
        col("product_id"),
        col("name").alias("product_name"),
        col("product_number"),
        col("color"),
        col("standard_cost"),
        col("list_price"),
        col("size"),
        col("weight"),
        col("product_line"),
        col("class"),
        col("style"),
        col("subcategory_name"),
        col("category_name"),
        col("sell_start_date"),
        col("sell_end_date"),
        col("discontinued_date")
    ).distinct()

dim_product.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("DimProduct")
print(f"OK: DimProduct ({dim_product.count():,} filas)")

StatementMeta(, 4545488f-96cd-4948-8e44-92251a8e5410, 4, Finished, Available, Finished, False)

OK: DimProduct (504 filas)


In [3]:
# Cell 4 - DimCustomer y DimTerritory
from pyspark.sql.functions import col, concat_ws

# DimCustomer - Customer + Person
df_customer = spark.read.format("delta").load(f"{SILVER_PATH}/Tables/silver_sales_customer")
df_person = spark.read.format("delta").load(f"{SILVER_PATH}/Tables/silver_person_person")

dim_customer = df_customer.join(
    df_person.select(
        col("business_entity_id"),
        col("first_name"),
        col("last_name"),
        col("middle_name"),
        col("person_type"),
        col("email_promotion")
    ),
    df_customer.person_id == df_person.business_entity_id,
    how="left"
).select(
    col("customer_id"),
    col("person_id"),
    col("store_id"),
    col("territory_id"),
    col("account_number"),
    col("first_name"),
    col("last_name"),
    col("middle_name"),
    col("person_type"),
    col("email_promotion")
).distinct()

dim_customer.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("DimCustomer")
print(f"OK: DimCustomer ({dim_customer.count():,} filas)")

# DimTerritory
df_territory = spark.read.format("delta").load(f"{SILVER_PATH}/Tables/silver_sales_salesterritory")

dim_territory = df_territory.select(
    col("territory_id"),
    col("name").alias("territory_name"),
    col("country_region_code"),
    col("group").alias("territory_group"),
    col("sales_ytd"),
    col("sales_last_year"),
    col("cost_ytd"),
    col("cost_last_year")
).distinct()

dim_territory.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("DimTerritory")
print(f"OK: DimTerritory ({dim_territory.count():,} filas)")

StatementMeta(, 4545488f-96cd-4948-8e44-92251a8e5410, 5, Finished, Available, Finished, False)

OK: DimCustomer (19,820 filas)
OK: DimTerritory (10 filas)


In [4]:
# Cell 5 - DimEmployee y DimVendor
from pyspark.sql.functions import col, concat_ws

# DimEmployee - EmployeeDepartmentHistory + EmployeePayHistory + Person
df_emp_dept = spark.read.format("delta").load(f"{SILVER_PATH}/Tables/silver_humanresources_employeedepartmenthistory")
df_emp_pay = spark.read.format("delta").load(f"{SILVER_PATH}/Tables/silver_humanresources_employeepayhistory")
df_person = spark.read.format("delta").load(f"{SILVER_PATH}/Tables/silver_person_person")
df_dept = spark.read.format("delta").load(f"{SILVER_PATH}/Tables/silver_humanresources_department")

# Ultimo departamento de cada empleado (end_date null = actual)
df_current_dept = df_emp_dept.filter(col("end_date").isNull()) \
    .select(
        col("business_entity_id"),
        col("department_id"),
        col("shift_id"),
        col("start_date").alias("dept_start_date")
    )

# Ultimo salario de cada empleado
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

window = Window.partitionBy("business_entity_id").orderBy(desc("rate_change_date"))
df_latest_pay = df_emp_pay.withColumn("rn", row_number().over(window)) \
    .filter(col("rn") == 1) \
    .select(
        col("business_entity_id"),
        col("rate").alias("current_rate"),
        col("pay_frequency"),
        col("rate_change_date")
    )

# Join todo
dim_employee = df_current_dept.join(
    df_person.select(
        col("business_entity_id"),
        col("first_name"),
        col("last_name"),
        col("middle_name")
    ),
    on="business_entity_id", how="left"
).join(
    df_latest_pay, on="business_entity_id", how="left"
).join(
    df_dept.select(col("department_id"), col("name").alias("department_name"), col("group_name")),
    on="department_id", how="left"
).select(
    col("business_entity_id").alias("employee_id"),
    col("first_name"),
    col("last_name"),
    col("middle_name"),
    col("department_id"),
    col("department_name"),
    col("group_name").alias("department_group"),
    col("shift_id"),
    col("dept_start_date"),
    col("current_rate"),
    col("pay_frequency"),
    col("rate_change_date")
).distinct()

dim_employee.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("DimEmployee")
print(f"OK: DimEmployee ({dim_employee.count():,} filas)")

# DimVendor
df_vendor = spark.read.format("delta").load(f"{SILVER_PATH}/Tables/silver_purchasing_vendor")

dim_vendor = df_vendor.select(
    col("business_entity_id").alias("vendor_id"),
    col("account_number"),
    col("name").alias("vendor_name"),
    col("credit_rating"),
    col("active_flag"),
    col("preferred_vendor_status")
).distinct()

dim_vendor.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("DimVendor")
print(f"OK: DimVendor ({dim_vendor.count():,} filas)")

StatementMeta(, 4545488f-96cd-4948-8e44-92251a8e5410, 6, Finished, Available, Finished, False)

OK: DimEmployee (290 filas)
OK: DimVendor (104 filas)


In [5]:
# Cell 6 - DimSalesPerson y DimDepartment
from pyspark.sql.functions import col

# DimSalesPerson
df_salesperson = spark.read.format("delta").load(f"{SILVER_PATH}/Tables/silver_sales_salesperson")
df_person = spark.read.format("delta").load(f"{SILVER_PATH}/Tables/silver_person_person")

dim_salesperson = df_salesperson.join(
    df_person.select(
        col("business_entity_id"),
        col("first_name"),
        col("last_name"),
        col("middle_name")
    ),
    on="business_entity_id", how="left"
).select(
    col("business_entity_id").alias("sales_person_id"),
    col("first_name"),
    col("last_name"),
    col("middle_name"),
    col("territory_id"),
    col("sales_quota"),
    col("bonus"),
    col("commission_pct"),
    col("sales_ytd"),
    col("sales_last_year")
).distinct()

dim_salesperson.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("DimSalesPerson")
print(f"OK: DimSalesPerson ({dim_salesperson.count():,} filas)")

# DimDepartment
df_dept = spark.read.format("delta").load(f"{SILVER_PATH}/Tables/silver_humanresources_department")

dim_department = df_dept.select(
    col("department_id"),
    col("name").alias("department_name"),
    col("group_name").alias("department_group")
).distinct()

dim_department.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("DimDepartment")
print(f"OK: DimDepartment ({dim_department.count():,} filas)")

StatementMeta(, 4545488f-96cd-4948-8e44-92251a8e5410, 7, Finished, Available, Finished, False)

OK: DimSalesPerson (17 filas)
OK: DimDepartment (16 filas)


In [6]:
# Cell 7 - FactSales
from pyspark.sql.functions import col, year, month, date_format, when
from pyspark.sql.types import IntegerType, LongType

df_header = spark.read.format("delta").load(f"{SILVER_PATH}/Tables/silver_sales_salesorderheader")
df_detail = spark.read.format("delta").load(f"{SILVER_PATH}/Tables/silver_sales_salesorderdetail")

df_detail_clean = df_detail.select(
    col("sales_order_id"),
    col("sales_order_detail_id"),
    col("product_id"),
    col("order_qty"),
    col("unit_price"),
    col("unit_price_discount"),
    col("line_total"),
    col("special_offer_id")
)

fact_sales = df_detail_clean.join(
    df_header.select(
        col("sales_order_id"),
        col("order_date"),
        col("due_date"),
        col("ship_date"),
        col("customer_id"),
        col("sales_person_id"),
        col("territory_id"),
        col("bill_to_address_id"),
        col("ship_to_address_id"),
        col("ship_method_id"),
        col("sub_total"),
        col("tax_amt"),
        col("freight"),
        col("total_due"),
        col("status"),
        col("online_order_flag"),
        col("purchase_order_number"),
        col("credit_card_id"),
        col("currency_rate_id")
    ),
    on="sales_order_id", how="inner"
).select(
    col("sales_order_id"),
    col("sales_order_detail_id"),
    col("order_date"),
    date_format(col("order_date"), "yyyyMMdd").cast(IntegerType()).alias("date_key"),
    year(col("order_date")).alias("order_year"),
    month(col("order_date")).alias("order_month"),
    col("due_date"),
    col("ship_date"),
    col("customer_id"),
    col("sales_person_id").cast(LongType()),
    col("territory_id"),
    col("product_id"),
    col("special_offer_id"),
    col("order_qty"),
    col("unit_price"),
    col("unit_price_discount"),
    col("line_total"),
    col("sub_total"),
    col("tax_amt"),
    col("freight"),
    col("total_due"),
    col("status"),
    when(col("online_order_flag") == True, "Online").otherwise("In-Store").alias("order_channel"),
    col("purchase_order_number"),
    col("credit_card_id"),
    col("currency_rate_id"),
    col("bill_to_address_id"),
    col("ship_to_address_id"),
    col("ship_method_id")
)

fact_sales.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("FactSales")
print(f"OK: FactSales ({fact_sales.count():,} filas)")

StatementMeta(, 4545488f-96cd-4948-8e44-92251a8e5410, 8, Finished, Available, Finished, False)

OK: FactSales (121,317 filas)


In [7]:
# Cell 8 - FactPurchasing
from pyspark.sql.functions import col, year, month, date_format
from pyspark.sql.types import IntegerType

df_po_header = spark.read.format("delta").load(f"{SILVER_PATH}/Tables/silver_purchasing_purchaseorderheader")
df_po_detail = spark.read.format("delta").load(f"{SILVER_PATH}/Tables/silver_purchasing_purchaseorderdetail")

df_detail_clean = df_po_detail.select(
    col("purchase_order_id"),
    col("purchase_order_detail_id"),
    col("product_id"),
    col("order_qty"),
    col("unit_price"),
    col("line_total"),
    col("received_qty"),
    col("rejected_qty"),
    col("stocked_qty"),
    col("due_date").alias("detail_due_date")
)

fact_purchasing = df_detail_clean.join(
    df_po_header.select(
        col("purchase_order_id"),
        col("order_date"),
        col("ship_date"),
        col("vendor_id"),
        col("ship_method_id"),
        col("sub_total"),
        col("tax_amt"),
        col("freight"),
        col("total_due"),
        col("status"),
        col("employee_id")
    ),
    on="purchase_order_id", how="inner"
).select(
    col("purchase_order_id"),
    col("purchase_order_detail_id"),
    col("order_date"),
    date_format(col("order_date"), "yyyyMMdd").cast(IntegerType()).alias("date_key"),
    year(col("order_date")).alias("order_year"),
    month(col("order_date")).alias("order_month"),
    col("ship_date"),
    col("vendor_id"),
    col("employee_id"),
    col("ship_method_id"),
    col("product_id"),
    col("order_qty"),
    col("unit_price"),
    col("line_total"),
    col("received_qty"),
    col("rejected_qty"),
    col("stocked_qty"),
    col("sub_total"),
    col("tax_amt"),
    col("freight"),
    col("total_due"),
    col("status")
)

fact_purchasing.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("FactPurchasing")
print(f"OK: FactPurchasing ({fact_purchasing.count():,} filas)")

StatementMeta(, 4545488f-96cd-4948-8e44-92251a8e5410, 9, Finished, Available, Finished, False)

OK: FactPurchasing (8,845 filas)


In [8]:
# Cell 9 - FactInventory y FactWorkOrder
from pyspark.sql.functions import col, date_format, year, month
from pyspark.sql.types import IntegerType

# FactInventory
df_inventory = spark.read.format("delta").load(f"{SILVER_PATH}/Tables/silver_production_productinventory")

fact_inventory = df_inventory.select(
    col("product_id"),
    col("location_id"),
    col("shelf"),
    col("bin"),
    col("quantity"),
    col("rowguid"),
    col("modified_date")
)

fact_inventory.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("FactInventory")
print(f"OK: FactInventory ({fact_inventory.count():,} filas)")

# FactWorkOrder
df_workorder = spark.read.format("delta").load(f"{SILVER_PATH}/Tables/silver_production_workorder")
df_workorder_routing = spark.read.format("delta").load(f"{SILVER_PATH}/Tables/silver_production_workorderrouting")

fact_workorder = df_workorder.join(
    df_workorder_routing.select(
        col("work_order_id"),
        col("product_id").alias("routing_product_id"),
        col("operation_sequence"),
        col("location_id"),
        col("actual_cost"),
        col("actual_resource_hrs")
    ),
    on="work_order_id", how="left"
).select(
    col("work_order_id"),
    col("product_id"),
    col("order_qty"),
    col("stocked_qty"),
    col("scrapped_qty"),
    col("start_date"),
    date_format(col("start_date"), "yyyyMMdd").cast(IntegerType()).alias("start_date_key"),
    year(col("start_date")).alias("start_year"),
    month(col("start_date")).alias("start_month"),
    col("end_date"),
    col("due_date"),
    col("scrap_reason_id"),
    col("operation_sequence"),
    col("location_id"),
    col("actual_cost"),
    col("actual_resource_hrs")
)

fact_workorder.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("FactWorkOrder")
print(f"OK: FactWorkOrder ({fact_workorder.count():,} filas)")

StatementMeta(, 4545488f-96cd-4948-8e44-92251a8e5410, 10, Finished, Available, Finished, False)

OK: FactInventory (1,069 filas)
OK: FactWorkOrder (97,097 filas)


In [9]:
# Cell 10 - FactEmployeeHistory
from pyspark.sql.functions import col, date_format, year, month, datediff, current_date
from pyspark.sql.types import IntegerType

df_emp_dept = spark.read.format("delta").load(f"{SILVER_PATH}/Tables/silver_humanresources_employeedepartmenthistory")
df_emp_pay = spark.read.format("delta").load(f"{SILVER_PATH}/Tables/silver_humanresources_employeepayhistory")

fact_emp_history = df_emp_dept.join(
    df_emp_pay.select(
        col("business_entity_id"),
        col("rate"),
        col("pay_frequency"),
        col("rate_change_date")
    ),
    on="business_entity_id", how="left"
).select(
    col("business_entity_id").alias("employee_id"),
    col("department_id"),
    col("shift_id"),
    col("start_date"),
    date_format(col("start_date"), "yyyyMMdd").cast(IntegerType()).alias("start_date_key"),
    col("end_date"),
    year(col("start_date")).alias("start_year"),
    month(col("start_date")).alias("start_month"),
    col("rate").alias("pay_rate"),
    col("pay_frequency"),
    col("rate_change_date")
)

fact_emp_history.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("FactEmployeeHistory")
print(f"OK: FactEmployeeHistory ({fact_emp_history.count():,} filas)")

StatementMeta(, 4545488f-96cd-4948-8e44-92251a8e5410, 11, Finished, Available, Finished, False)

OK: FactEmployeeHistory (334 filas)


In [11]:
# Cell 11 - FactNewItems
from pyspark.sql.functions import col, year, lit
from functools import reduce

# Leer desde metastore
df_factsales = spark.table("FactSales")
df_product = spark.table("DimProduct")

products_by_year = df_factsales.select(
    col("product_id"),
    col("order_year")
).distinct()

# Verificar años disponibles
print("Años disponibles:")
products_by_year.groupBy("order_year").count().orderBy("order_year").show()

years = [row.order_year for row in products_by_year.select("order_year").distinct().orderBy("order_year").collect()]
print(f"Anos: {years}")

rows = []
for base_year in years:
    for comp_year in years:
        if comp_year <= base_year:
            continue
        
        base_products = products_by_year.filter(col("order_year") == base_year).select("product_id")
        
        new_products = products_by_year.filter(
            (col("order_year") > base_year) & (col("order_year") <= comp_year)
        ).select("product_id").distinct()
        
        truly_new = new_products.subtract(base_products)
        
        truly_new = truly_new \
            .withColumn("base_year", lit(base_year)) \
            .withColumn("comp_year", lit(comp_year)) \
            .withColumn("is_new_item", lit(1))
        
        rows.append(truly_new)

fact_new_items = reduce(lambda a, b: a.union(b), rows)

revenue_by_product_year = df_factsales.groupBy("product_id", "order_year") \
    .sum("line_total") \
    .withColumnRenamed("sum(line_total)", "revenue") \
    .withColumnRenamed("order_year", "sales_year")

fact_new_items_enriched = fact_new_items.join(
    df_product.select("product_id", "product_name", "category_name", "subcategory_name", "list_price", "standard_cost"),
    on="product_id", how="left"
).join(
    revenue_by_product_year,
    (fact_new_items.product_id == revenue_by_product_year.product_id) &
    (fact_new_items.comp_year == revenue_by_product_year.sales_year),
    how="left"
).drop(revenue_by_product_year.product_id).drop("sales_year")

fact_new_items_enriched.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("FactNewItems")
print(f"OK: FactNewItems ({fact_new_items_enriched.count():,} filas)")

print("\nResumen por combinación de años:")
fact_new_items_enriched.groupBy("base_year", "comp_year").count().orderBy("base_year", "comp_year").show()

StatementMeta(, 4545488f-96cd-4948-8e44-92251a8e5410, 13, Finished, Available, Finished, False)

Años disponibles:
+----------+-----+
|order_year|count|
+----------+-----+
|      2022|   60|
|      2023|  132|
|      2024|  238|
|      2025|  180|
+----------+-----+

Anos: [2022, 2023, 2024, 2025]
OK: FactNewItems (752 filas)

Resumen por combinación de años:
+---------+---------+-----+
|base_year|comp_year|count|
+---------+---------+-----+
|     2022|     2023|   72|
|     2022|     2024|  206|
|     2022|     2025|  206|
|     2023|     2024|  134|
|     2023|     2025|  134|
+---------+---------+-----+



In [12]:
# Cell 12 - Verificación final Gold layer
gold_tables = [
    "DimDate", "DimProduct", "DimCustomer", "DimTerritory",
    "DimSalesPerson", "DimEmployee", "DimVendor", "DimDepartment",
    "FactSales", "FactPurchasing", "FactInventory", 
    "FactWorkOrder", "FactEmployeeHistory", "FactNewItems"
]

print("=" * 50)
print("GOLD LAYER - RESUMEN FINAL")
print("=" * 50)

total_rows = 0
for table in gold_tables:
    try:
        df = spark.table(table)
        rows = df.count()
        cols = len(df.columns)
        total_rows += rows
        print(f"✅ {table:<25} {rows:>10,} filas | {cols:>3} cols")
    except Exception as e:
        print(f"❌ {table:<25} ERROR: {str(e)[:50]}")

print("=" * 50)
print(f"Total filas Gold: {total_rows:,}")
print(f"Total tablas: {len(gold_tables)}")

StatementMeta(, 4545488f-96cd-4948-8e44-92251a8e5410, 14, Finished, Available, Finished, False)

GOLD LAYER - RESUMEN FINAL
✅ DimDate                        6,209 filas |  21 cols
✅ DimProduct                       504 filas |  16 cols
✅ DimCustomer                   19,820 filas |  10 cols
✅ DimTerritory                      10 filas |   8 cols
✅ DimSalesPerson                    17 filas |  10 cols
✅ DimEmployee                      290 filas |  12 cols
✅ DimVendor                        104 filas |   6 cols
✅ DimDepartment                     16 filas |   3 cols
✅ FactSales                    121,317 filas |  29 cols
✅ FactPurchasing                 8,845 filas |  22 cols
✅ FactInventory                  1,069 filas |   7 cols
✅ FactWorkOrder                 97,097 filas |  16 cols
✅ FactEmployeeHistory              334 filas |  11 cols
✅ FactNewItems                     752 filas |  10 cols
Total filas Gold: 256,384
Total tablas: 14


In [13]:
# Cell 13 - Enriquecer DimProduct con Manufacturer y Brand
from pyspark.sql.functions import col, lit

df_product = spark.table("DimProduct")

dim_product_enriched = df_product \
    .withColumn("manufacturer", lit("AdventureWorks")) \
    .withColumn("brand", lit("AdventureWorks"))

dim_product_enriched.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("DimProduct")
print(f"OK: DimProduct enriquecido ({dim_product_enriched.count():,} filas)")
print("Manufacturer y Brand agregados: AdventureWorks")

StatementMeta(, 4545488f-96cd-4948-8e44-92251a8e5410, 15, Finished, Available, Finished, False)

OK: DimProduct enriquecido (504 filas)
Manufacturer y Brand agregados: AdventureWorks


In [14]:
# Cell 14 - Crear FactCompetitorSales y DimCompetitor
import pandas as pd
import numpy as np
from pyspark.sql.functions import col, lit
from datetime import datetime, timedelta

np.random.seed(42)

# Definir competidores
competitors = [
    {"competitor_id": 1, "manufacturer": "Trek Industries", "brand": "Trek"},
    {"competitor_id": 2, "manufacturer": "Giant Sports", "brand": "Giant"}
]

# Categorías que venden (solo Bikes y Components)
categories = ["Bikes", "Components"]

# Generar ventas semanales por competidor, categoría y año
# Usamos ISO weeks de nuestros datos: 2022-2025
date_range = pd.date_range(start='2022-01-03', end='2025-06-30', freq='W-MON')

rows = []
for comp in competitors:
    for category in categories:
        # Base revenue semanal diferente por competidor y categoría
        if comp["brand"] == "Trek":
            base_revenue = 800000 if category == "Bikes" else 200000
        else:
            base_revenue = 600000 if category == "Bikes" else 150000
            
        for date in date_range:
            iso = date.isocalendar()
            # Agregar variación aleatoria y estacionalidad
            seasonality = 1 + 0.3 * np.sin(2 * np.pi * date.month / 12)
            noise = np.random.uniform(0.85, 1.15)
            trend = 1 + 0.05 * (date.year - 2022)  # 5% crecimiento anual
            
            weekly_revenue = base_revenue * seasonality * noise * trend
            weekly_qty = int(weekly_revenue / (500 if category == "Bikes" else 150))
            
            rows.append({
                "competitor_id": comp["competitor_id"],
                "manufacturer": comp["manufacturer"],
                "brand": comp["brand"],
                "category_name": category,
                "order_date": date,
                "date_key": int(date.strftime('%Y%m%d')),
                "order_year": date.year,
                "order_month": date.month,
                "iso_year": iso[0],
                "iso_week": iso[1],
                "iso_year_week": f"{iso[0]}-W{iso[1]:02d}",
                "order_qty": weekly_qty,
                "revenue": round(weekly_revenue, 2)
            })

df_competitor = pd.DataFrame(rows)
fact_competitor = spark.createDataFrame(df_competitor)

fact_competitor.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("FactCompetitorSales")
print(f"OK: FactCompetitorSales ({fact_competitor.count():,} filas)")

print("\nRevenue total por competidor y categoría:")
fact_competitor.groupBy("brand", "category_name") \
    .sum("revenue") \
    .withColumnRenamed("sum(revenue)", "total_revenue") \
    .orderBy("brand", "category_name") \
    .show()

StatementMeta(, 4545488f-96cd-4948-8e44-92251a8e5410, 16, Finished, Available, Finished, False)

OK: FactCompetitorSales (732 filas)

Revenue total por competidor y categoría:
+-----+-------------+--------------------+
|brand|category_name|       total_revenue|
+-----+-------------+--------------------+
|Giant|        Bikes|1.2124673821000001E8|
|Giant|   Components|2.9582744299999997E7|
| Trek|        Bikes|      1.5859746286E8|
| Trek|   Components|       4.006753401E7|
+-----+-------------+--------------------+



In [16]:
# Cell 15 - Verificar Market Share (corregido)
from pyspark.sql.functions import col, sum as spark_sum, round as spark_round, lit
from pyspark.sql.window import Window

# Revenue AdventureWorks por categoría
df_aw = spark.table("FactSales").join(
    spark.table("DimProduct").select("product_id", "category_name"),
    on="product_id", how="left"
).filter(col("category_name").isin(["Bikes", "Components"])) \
.groupBy("category_name") \
.agg(spark_sum("line_total").alias("revenue")) \
.withColumn("brand", lit("AdventureWorks")) \
.select("brand", "category_name", "revenue")

# Revenue competidores
df_comp = spark.table("FactCompetitorSales") \
    .groupBy("brand", "category_name") \
    .agg(spark_sum("revenue").alias("revenue")) \
    .select("brand", "category_name", "revenue")

# Unir
df_total = df_aw.union(df_comp)

# Calcular share
window = Window.partitionBy("category_name")
df_share = df_total.withColumn(
    "market_total", spark_sum("revenue").over(window)
).withColumn(
    "market_share_pct", spark_round(col("revenue") / col("market_total") * 100, 2)
).orderBy("category_name", "brand")

df_share.show()

StatementMeta(, 4545488f-96cd-4948-8e44-92251a8e5410, 18, Finished, Available, Finished, False)

+--------------+-------------+--------------------+--------------------+----------------+
|         brand|category_name|             revenue|        market_total|market_share_pct|
+--------------+-------------+--------------------+--------------------+----------------+
|AdventureWorks|        Bikes| 9.465117270469621E7|3.7449537377469623E8|           25.27|
|         Giant|        Bikes|1.2124673821000001E8|3.7449537377469623E8|           32.38|
|          Trek|        Bikes|      1.5859746286E8|3.7449537377469623E8|           42.35|
|AdventureWorks|   Components|1.1802593286429755E7| 8.145287159642975E7|           14.49|
|         Giant|   Components|2.9582744299999997E7| 8.145287159642975E7|           36.32|
|          Trek|   Components|       4.006753401E7| 8.145287159642975E7|           49.19|
+--------------+-------------+--------------------+--------------------+----------------+



In [1]:
# Cell 18 - DimManufacturer
import pandas as pd

manufacturers = [
    {"manufacturer_id": 1, "manufacturer_name": "AdventureWorks", "brand": "AdventureWorks", "is_competitor": 0},
    {"manufacturer_id": 2, "manufacturer_name": "Trek Industries", "brand": "Trek", "is_competitor": 1},
    {"manufacturer_id": 3, "manufacturer_name": "Giant Sports", "brand": "Giant", "is_competitor": 1}
]

df_mfr = pd.DataFrame(manufacturers)
dim_manufacturer = spark.createDataFrame(df_mfr)

dim_manufacturer.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("DimManufacturer")
print(f"OK: DimManufacturer ({dim_manufacturer.count():,} filas)")
dim_manufacturer.show()

StatementMeta(, 7ef7293e-7d2e-466f-8623-59bee49b033b, 3, Finished, Available, Finished, False)

OK: DimManufacturer (3 filas)
+---------------+-----------------+--------------+-------------+
|manufacturer_id|manufacturer_name|         brand|is_competitor|
+---------------+-----------------+--------------+-------------+
|              1|   AdventureWorks|AdventureWorks|            0|
|              2|  Trek Industries|          Trek|            1|
|              3|     Giant Sports|         Giant|            1|
+---------------+-----------------+--------------+-------------+



In [4]:
# Cell 17 - Agregar category_id a FactCompetitorSales
from pyspark.sql.functions import col, when

df_competitor = spark.table("FactCompetitorSales")
df_category = spark.read.format("delta").load(f"{SILVER_PATH}/Tables/silver_production_productcategory")

# Mapear category_name a category_id
df_cat_map = df_category.select(
    col("product_category_id").alias("category_id"),
    col("name").alias("category_name")
)

fact_competitor_updated = df_competitor.join(
    df_cat_map, on="category_name", how="left"
)

fact_competitor_updated.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("FactCompetitorSales")
print(f"OK: FactCompetitorSales actualizada ({fact_competitor_updated.count():,} filas)")
print("\nVerificación category_id:")
fact_competitor_updated.select("brand", "category_name", "category_id").distinct().show()

StatementMeta(, 7ef7293e-7d2e-466f-8623-59bee49b033b, 6, Finished, Available, Finished, False)

OK: FactCompetitorSales actualizada (732 filas)

Verificación category_id:
+-----+-------------+-----------+
|brand|category_name|category_id|
+-----+-------------+-----------+
| Trek|   Components|          2|
|Giant|   Components|          2|
| Trek|        Bikes|          1|
|Giant|        Bikes|          1|
+-----+-------------+-----------+



In [7]:
# Cell 19 - Final
from pyspark.sql.functions import col, lit

# FactCompetitorSales - limpiar y reagregar columnas
df_competitor = spark.table("FactCompetitorSales")

# Dropear columnas que ya existen para evitar duplicados
cols_to_drop = [c for c in ["manufacturer_id", "category_id"] if c in df_competitor.columns]
if cols_to_drop:
    df_competitor = df_competitor.drop(*cols_to_drop)

# Agregar category_id
df_category = spark.read.format("delta").load(f"{SILVER_PATH}/Tables/silver_production_productcategory")
df_cat_map = df_category.select(
    col("product_category_id").alias("category_id"),
    col("name").alias("category_name")
)

# Agregar manufacturer_id
dim_mfr = spark.table("DimManufacturer")

fact_competitor_updated = df_competitor \
    .join(df_cat_map, on="category_name", how="left") \
    .join(dim_mfr.select("manufacturer_id", "brand"), on="brand", how="left")

fact_competitor_updated.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("FactCompetitorSales")
print(f"OK: FactCompetitorSales ({fact_competitor_updated.count():,} filas)")
print(f"Columnas: {fact_competitor_updated.columns}")

print("\nVerificación:")
fact_competitor_updated.select("brand", "manufacturer_id", "category_name", "category_id").distinct().show()

StatementMeta(, 7ef7293e-7d2e-466f-8623-59bee49b033b, 9, Finished, Available, Finished, False)

OK: FactCompetitorSales (732 filas)
Columnas: ['brand', 'category_name', 'competitor_id', 'manufacturer', 'order_date', 'date_key', 'order_year', 'order_month', 'iso_year', 'iso_week', 'iso_year_week', 'order_qty', 'revenue', 'category_id', 'manufacturer_id']

Verificación:
+-----+---------------+-------------+-----------+
|brand|manufacturer_id|category_name|category_id|
+-----+---------------+-------------+-----------+
| Trek|              2|   Components|          2|
| Trek|              2|        Bikes|          1|
|Giant|              3|        Bikes|          1|
|Giant|              3|   Components|          2|
+-----+---------------+-------------+-----------+

